In [ ]:
#mount and install
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install -q chronos-forecasting

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.6/80.6 kB 5.2 MB/s eta 0:00:00


In [ ]:

import sys, os
sys.path.append('/content/drive/MyDrive/Colab Notebooks/FreshRetailNet Demand Forecasting/src')

import numpy as np
import pandas as pd
import config as cf, data, predictions

df = data.load()
print(len(df), 'rows')

4850000 rows


In [ ]:
#load the pretrained model (zero-shot: no training happens)
from chronos import Chronos2Pipeline
pipeline = Chronos2Pipeline.from_pretrained("amazon/chronos-2", device_map="cuda")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:121: UserWarning: 
Access to the secret `HF_TOKEN` has not been granted on this notebook.
You will not be requested again.
Please restart the session if you want to be prompted again.
  warnings.warn(


config.json:   0%|          | 0.00/1.07k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  478MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/170 [00:00<?, ?it/s]

In [ ]:
# the runner
def run_chronos(train_df, futr_df, input_version, split, batch_series=2000):

    ctx = train_df[['unique_id', 'ds', 'y'] + cf.EXOG]     # statics not supported
    ids = ctx['unique_id'].unique()
    parts = []

    for i in range(0, len(ids), batch_series):
        b = ids[i:i + batch_series]
        p = pipeline.predict_df(
            ctx[ctx.unique_id.isin(b)],
            future_df=futr_df[futr_df.unique_id.isin(b)],
            prediction_length=cf.HORIZON,
            quantile_levels=[0.1, 0.5, 0.9],
            id_column='unique_id', timestamp_column='ds', target='y')
        parts.append(p)
        print(f'  {min(i + batch_series, len(ids))}/{len(ids)}')

    pred = pd.concat(parts, ignore_index=True).rename(columns={'predictions': 'prediction'})
    pred[['store_id', 'product_id']] = pred['unique_id'].str.split('_', expand=True).astype(int)
    pred['dt'] = pd.to_datetime(pred['ds']).dt.strftime('%Y-%m-%d')

    predictions.save(pred[['store_id', 'product_id', 'dt', 'prediction']],
                     'Chronos2', input_version, split)

In [ ]:
# real sales, validation week
train_df, futr_df = data.splittbl(df, 'raw', 'val')
run_chronos(train_df, futr_df, 'raw', 'val')

  2000/50000
  4000/50000
  6000/50000
  8000/50000
  10000/50000
  12000/50000
  14000/50000
  16000/50000
  18000/50000
  20000/50000
  22000/50000
  24000/50000
  26000/50000
  28000/50000
  30000/50000
  32000/50000
  34000/50000
  36000/50000
  38000/50000
  40000/50000
  42000/50000
  44000/50000
  46000/50000
  48000/50000
  50000/50000
[val] 350,000 rows : Chronos2__raw


In [ ]:
# recovered demand, validation week
train_df, predict_df = data.splittbl(df, 'recovered', 'val')
run_chronos(train_df, predict_df, 'recovered', 'val')

  2000/50000
  4000/50000
  6000/50000
  8000/50000
  10000/50000
  12000/50000
  14000/50000
  16000/50000
  18000/50000
  20000/50000
  22000/50000
  24000/50000
  26000/50000
  28000/50000
  30000/50000
  32000/50000
  34000/50000
  36000/50000
  38000/50000
  40000/50000
  42000/50000
  44000/50000
  46000/50000
  48000/50000
  50000/50000
[val] 350,000 rows : Chronos2__recovered


In [ ]:
# real sales, test week
train_df, predict_df = data.splittbl(df, 'raw', 'test')
run_chronos(train_df, predict_df, 'raw', 'test')

  2000/50000
  4000/50000
  6000/50000
  8000/50000
  10000/50000
  12000/50000
  14000/50000
  16000/50000
  18000/50000
  20000/50000
  22000/50000
  24000/50000
  26000/50000
  28000/50000
  30000/50000
  32000/50000
  34000/50000
  36000/50000
  38000/50000
  40000/50000
  42000/50000
  44000/50000
  46000/50000
  48000/50000
  50000/50000
[test] 350,000 rows : Chronos2__raw


In [ ]:
# recovered demand, test week
train_df, predict_df = data.splittbl(df, 'recovered', 'test')
run_chronos(train_df, predict_df, 'recovered', 'test')

  2000/50000
  4000/50000
  6000/50000
  8000/50000
  10000/50000
  12000/50000
  14000/50000
  16000/50000
  18000/50000
  20000/50000
  22000/50000
  24000/50000
  26000/50000
  28000/50000
  30000/50000
  32000/50000
  34000/50000
  36000/50000
  38000/50000
  40000/50000
  42000/50000
  44000/50000
  46000/50000
  48000/50000
  50000/50000
[test] 350,000 rows : Chronos2__recovered


In [ ]:
# check all four landed
import os
for folder, split in [(cf.PRED_VAL, 'val'), (cf.PRED_TEST, 'test')]:
    for iv in ['raw', 'recovered']:
        f = f'{folder}/Chronos2__{iv}.parquet'
        p = pd.read_parquet(f)
        print(f'{split:5s} {iv:10s} rows {len(p):,} | mean {p.prediction.mean():.4f} '
              f'| range {p.prediction.min():.2f}-{p.prediction.max():.2f}')

val   raw        rows 350,000 | mean 1.0493 | range 0.00-38.90
val   recovered  rows 350,000 | mean 1.2265 | range 0.00-45.85
test  raw        rows 350,000 | mean 1.1092 | range 0.00-46.20
test  recovered  rows 350,000 | mean 1.2752 | range 0.00-45.96


In [ ]:
# the results table
import importlib, evaluate
importlib.reload(evaluate)

for split in ['val', 'test']:
    long, wide = evaluate.results(split)
    long.to_csv(f'{cf.EVALUATION}/results_{split}.csv', index=False)
    print(f'\n===== {split.upper()} =====')
    print(wide.to_string())


===== VAL =====
input             raw  recovered    gain
model                                   
LightGBM       0.3180     0.2884  0.0296
TFT            0.3263     0.2895  0.0368
TimesFM        0.3238     0.2974  0.0264
Chronos2       0.3315     0.3050  0.0265
Informer       0.3174     0.3091  0.0083
TiDE           0.3386     0.3133  0.0253
PatchTST       0.3366     0.3147  0.0219
CrostonSBA     0.3572     0.3272  0.0300
LSTM           0.3584     0.3396  0.0188
DLinear        0.3673     0.3397  0.0276
SeasonalNaive  0.3976     0.3739  0.0237

===== TEST =====
input             raw  recovered    gain
model                                   
TFT            0.2947     0.2865  0.0082
TimesFM        0.3045     0.2895  0.0150
TFT_baseline   0.3147     0.2896  0.0251
LightGBM       0.3230     0.2951  0.0279
Chronos2       0.3200     0.2993  0.0207
TiDE           0.3162     0.3093  0.0069
PatchTST       0.3150     0.3116  0.0034
LSTM           0.3139     0.3138  0.0001
CrostonSBA     0.3391 

In [ ]:
# --- environment record ---
import platform, subprocess, sys
from importlib.metadata import version, PackageNotFoundError

print("OS      :", platform.platform())
print("Python  :", sys.version.split()[0])

try:
    import torch
    print("torch   :", torch.__version__, "| CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU     :", torch.cuda.get_device_name(0),
              "|", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
except ImportError:
    print("torch   : not installed")

print("CPU     :", subprocess.run("nproc", capture_output=True, text=True).stdout.strip(), "cores")
print("RAM     :", subprocess.run("free -g | awk 'NR==2{print $2}'", shell=True,
                                  capture_output=True, text=True).stdout.strip(), "GB")

PKGS = ["neuralforecast", "mlforecast", "lightgbm", "statsforecast", "optuna",
        "timesfm", "chronos-forecasting", "pypots", "pandas", "numpy", "pyarrow",
        "scikit-learn", "pytorch-lightning"]
print()
for p in PKGS:
    try:
        print(f"{p:22s} {version(p)}")
    except PackageNotFoundError:
        print(f"{p:22s} -")

OS      : Linux-6.6.122+-x86_64-with-glibc2.35
Python  : 3.12.13
torch   : 2.11.0+cu128 | CUDA available: True
GPU     : Tesla T4 | 15.6 GB
CPU     : 8 cores
RAM     : 50 GB

neuralforecast         -
mlforecast             -
lightgbm               4.6.0
statsforecast          -
optuna                 -
timesfm                -
chronos-forecasting    2.3.1
pypots                 -
pandas                 2.2.2
numpy                  2.0.2
pyarrow                18.1.0
scikit-learn           1.6.1
pytorch-lightning      -
